In [1]:
import os
from google.colab import drive

# Mount a disk
gdrive_dir = '/content/drive/'
drive.mount(gdrive_dir, force_remount=True)

# Look for two folders
folders = [
    '/content/drive/MyDrive/!corruption/',
    '/content/drive/MyDrive/!records/',
    '/content/drive/MyDrive/!transcripts/',
    '/content/drive/MyDrive/!keys/'
]

# Check them
for folder in folders:
    print(f"\n=== Files in: {folder} ===")
    for dirname, _, filenames in os.walk(folder):
        for filename in filenames:
            print(os.path.join(dirname, filename))

Mounted at /content/drive/

=== Files in: /content/drive/MyDrive/!corruption/ ===

=== Files in: /content/drive/MyDrive/!records/ ===

=== Files in: /content/drive/MyDrive/!transcripts/ ===

=== Files in: /content/drive/MyDrive/!keys/ ===
/content/drive/MyDrive/!keys/assemblyai.txt
/content/drive/MyDrive/!keys/openai.txt


In [2]:
pip install -q yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.6 MB/s eta 0:00:00


In [4]:
pip install ffmpeg

  Preparing metadata (setup.py) ... done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6083 sha256=9fb0c4ab197726562c1347b0b1b061454d5e58ca54e621d99bef6d85556ade6c
  Stored in directory: /root/.cache/pip/wheels/26/21/0c/c26e09dff860a9071683e279445262346e008a9a1d2142c4ad
Successfully built ffmpeg


In [ ]:
# =========================================================
# INSTALL / UPDATE YT-DLP + EJS + FFMPEG + DENO FOR COLAB
# =========================================================

!pip install -U "yt-dlp[default]" yt-dlp-ejs
!apt-get update -qq
!apt-get install -y ffmpeg curl unzip

!curl -fsSL https://deno.land/install.sh | sh

import os
os.environ["PATH"] += ":/root/.deno/bin"

!which deno
!deno --version
!python -m yt_dlp --version

In [17]:
!python -m yt_dlp \
  --cookies "/content/drive/MyDrive/!records/cookies.txt" \
  --js-runtimes "deno:/root/.deno/bin/deno" \
  --remote-components ejs:github \
  --list-formats "https://www.youtube.com/watch?v=CHuztvrUp64"

[youtube] Extracting URL: https://www.youtube.com/watch?v=CHuztvrUp64
[youtube] CHuztvrUp64: Downloading webpage
[youtube] CHuztvrUp64: Downloading tv downgraded player API JSON
[youtube] CHuztvrUp64: Downloading player 9fc68080-main
[youtube] [jsc:deno] Solving JS challenges using deno
[youtube] CHuztvrUp64: Downloading m3u8 information
[info] Available formats for CHuztvrUp64:
ID  EXT   RESOLUTION FPS CH │   FILESIZE   TBR PROTO │ VCODEC       VBR ACODEC      ABR ASR MORE INFO
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────
sb3 mhtml 48x27        0    │                  mhtml │ images                               storyboard
sb2 mhtml 80x45        0    │                  mhtml │ images                               storyboard
sb1 mhtml 160x90       0    │                  mhtml │ images                               storyboard
sb0 mhtml 320x180      0    │                  mhtml │ images                               

In [9]:
!apt-get install -y nodejs npm

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  gyp javascript-common libc-ares2 libjs-events libjs-highlight.js
  libjs-inherits libjs-is-typedarray libjs-psl libjs-source-map
  libjs-sprintf-js libjs-typedarray-to-buffer libnode-dev libnode72
  libnotify-bin libnotify4 libuv1-dev node-abab node-abbrev node-agent-base
  node-ansi-regex node-ansi-styles node-ansistyles node-aproba node-archy
  node-are-we-there-yet node-argparse node-arrify node-asap node-asynckit
  node-balanced-match node-brace-expansion node-builtins node-cacache
  node-chalk node-chownr node-clean-yaml-object node-cli-table node-clone
  node-color-convert node-color-name node-colors node-columnify
  node-combined-stream node-commander node-console-control-strings
  node-copy-concurrently node-core-util-is node-coveralls node-cssom
  node-cssstyle node-debug node-decompress-response node-defaults
  node-delayed-st

In [ ]:
import os
import re
import time
import shutil
import pandas as pd
from yt_dlp import YoutubeDL


# =========================================================
# PATHS FOR GOOGLE COLAB + GOOGLE DRIVE
# =========================================================

BASE_DIR = "/content/drive/MyDrive/!records/"

INPUT_FILE = os.path.join(
    BASE_DIR,
    "base.xlsx"
)

INPUT_SHEET = "videos"

URL_COLUMN = "video_url"
TASK_COLUMN = "task"
LANG_COLUMN = "lang"

AUDIO_OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "downloads_mp3_low"
)

LOG_FILE = os.path.join(
    BASE_DIR,
    "download_log_task_mp3_low.xlsx"
)

COOKIES_FILE = os.path.join(
    BASE_DIR,
    "cookies.txt"
)

SLEEP_SECONDS = 1


# =========================================================
# LOW QUALITY AUDIO SETTINGS
# =========================================================

MP3_QUALITY = "64"      # 64 kbps
AUDIO_RATE = "16000"    # 16 kHz
AUDIO_CHANNELS = "1"    # mono


# =========================================================
# YT-DLP / YOUTUBE SETTINGS
# =========================================================

DENO_PATH = "/root/.deno/bin/deno"

YOUTUBE_PLAYER_CLIENTS = [
    "web"
]


# =========================================================
# HELPERS
# =========================================================

def clean_filename(text):
    """
    Clean filename from invalid symbols.
    """
    if text is None or pd.isna(text):
        return "unknown"

    text = str(text).strip()

    text = re.sub(r'[\\/*?:"<>|]', "_", text)
    text = re.sub(r"\s+", " ", text)

    return text[:180]


def clean_lang(text):
    """
    Clean language code from the lang column.
    """
    if text is None or pd.isna(text):
        return "unknown"

    text = str(text).strip().lower()

    text = re.sub(
        r"[^a-zA-Zа-яА-ЯіІїЇєЄґҐ0-9_-]",
        "_",
        text
    )

    text = re.sub(r"_+", "_", text)
    text = text.strip("_")

    if not text:
        return "unknown"

    return text


def ensure_dirs():
    """
    Create output directories if not exist.
    """
    os.makedirs(AUDIO_OUTPUT_DIR, exist_ok=True)


def check_cookies_file():
    """
    Check whether cookies.txt exists and is not empty.
    """
    print("\nChecking cookies file:")
    print(COOKIES_FILE)

    if not os.path.exists(COOKIES_FILE):
        raise FileNotFoundError(
            f"cookies.txt not found: {COOKIES_FILE}"
        )

    file_size = os.path.getsize(COOKIES_FILE)

    if file_size == 0:
        raise ValueError(
            f"cookies.txt is empty: {COOKIES_FILE}"
        )

    print(f"cookies.txt found. Size: {file_size} bytes")


def check_deno_runtime():
    """
    Check whether Deno runtime is available.
    """
    print("\nChecking Deno runtime:")
    print(DENO_PATH)

    if os.path.exists(DENO_PATH):
        runtime_path = DENO_PATH
    else:
        runtime_path = shutil.which("deno")

    if runtime_path is None:
        raise RuntimeError(
            "Deno runtime not found. Run the installation cell first."
        )

    print(f"Deno found at: {runtime_path}")

    return runtime_path


def load_tasks():
    """
    Load Excel and keep only rows where task == 1.
    """

    print(f"\nLoading file: {INPUT_FILE}")

    df = pd.read_excel(
        INPUT_FILE,
        sheet_name=INPUT_SHEET
    )

    required_columns = [
        URL_COLUMN,
        TASK_COLUMN,
        LANG_COLUMN,
    ]

    for column in required_columns:
        if column not in df.columns:
            raise ValueError(
                f"Column '{column}' not found in sheet '{INPUT_SHEET}'"
            )

    task_numeric = pd.to_numeric(
        df[TASK_COLUMN],
        errors="coerce"
    )

    filtered_df = df[task_numeric == 1].copy()

    print("\nUnique raw task values:")
    print(df[TASK_COLUMN].drop_duplicates().tolist())

    print("\nUnique raw lang values:")
    print(df[LANG_COLUMN].drop_duplicates().tolist())

    print(f"\nTotal rows in sheet: {len(df)}")
    print(
        f"Rows selected for processing "
        f"where {TASK_COLUMN} == 1: {len(filtered_df)}"
    )
    return filtered_df

def get_common_ydl_options():
    """
    Common yt-dlp options for YouTube authentication,
    Deno EJS challenge solving and retry logic.
    """
    return {
        # -----------------------------------------
        # YOUTUBE AUTH
        # -----------------------------------------
        "cookiefile": COOKIES_FILE,
        # -----------------------------------------
        # JAVASCRIPT RUNTIME FOR YOUTUBE N CHALLENGE
        # Equivalent to:
        # --js-runtimes deno:/root/.deno/bin/deno
        # -----------------------------------------
        "js_runtimes": {
            "deno": {
                "path": DENO_PATH
            }
        },
        # -----------------------------------------
        # ALLOW EJS REMOTE COMPONENTS
        # Equivalent to:
        # --remote-components ejs:github
        # -----------------------------------------
        "remote_components": [
            "ejs:github"
        ],
        # -----------------------------------------
        # YOUTUBE CLIENT
        # Do not use ios/android with cookies
        # -----------------------------------------
        "extractor_args": {
            "youtube": {
                "player_client": ["web"],
            }
        },
        # -----------------------------------------
        # RESILIENCE
        # -----------------------------------------
        "retries": 5,
        "fragment_retries": 5,
        "extractor_retries": 5,
        "ignoreerrors": False,
    }


def get_video_info(url):
    """
    Fetch video metadata.
    """

    opts = {
        "quiet": True,
        "skip_download": True,
        "noplaylist": True,
        "extract_flat": False,

        **get_common_ydl_options(),
    }

    with YoutubeDL(opts) as ydl:
        info = ydl.extract_info(
            url,
            download=False
        )

    return info


def download_mp3_low_quality(url, file_stub):
    """
    Download low-quality mp3 optimized for transcription.
    """

    outtmpl = os.path.join(
        AUDIO_OUTPUT_DIR,
        f"{file_stub}.%(ext)s"
    )

    ydl_opts = {
        # -----------------------------------------
        # FORMAT
        # -----------------------------------------

        "format": "bestaudio/best",

        # -----------------------------------------
        # OUTPUT
        # -----------------------------------------

        "outtmpl": outtmpl,
        "noplaylist": True,

        # -----------------------------------------
        # LOGGING
        # -----------------------------------------

        "quiet": False,
        "verbose": False,

        # -----------------------------------------
        # YOUTUBE AUTH + DENO EJS
        # -----------------------------------------

        **get_common_ydl_options(),

        # -----------------------------------------
        # AUDIO CONVERSION
        # -----------------------------------------

        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": MP3_QUALITY,
            }
        ],

        "postprocessor_args": [
            "-ar", AUDIO_RATE,
            "-ac", AUDIO_CHANNELS,
        ],
    }

    with YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])


# =========================================================
# MAIN
# =========================================================

def main():

    ensure_dirs()
    check_cookies_file()
    check_deno_runtime()

    task_df = load_tasks()

    log_rows = []

    total = len(task_df)

    for counter, (_, row_data) in enumerate(
        task_df.iterrows(),
        start=1
    ):

        url = row_data[URL_COLUMN]

        lang = clean_lang(
            row_data.get(
                LANG_COLUMN,
                "unknown"
            )
        )

        print("\n" + "=" * 70)
        print(f"[{counter}/{total}] Processing:")
        print(url)

        log_row = {
            "url": url,
            "task": row_data.get(TASK_COLUMN, ""),
            "lang": lang,
            "title": "",
            "title_stub": "",
            "file_stub": "",
            "audio_status": "",
            "audio_file_type": "mp3",
            "audio_bitrate_kbps": MP3_QUALITY,
            "audio_sample_rate": AUDIO_RATE,
            "audio_channels": AUDIO_CHANNELS,
            "error_audio": "",
        }

        try:

            # -----------------------------------------
            # VIDEO INFO
            # -----------------------------------------

            info = get_video_info(url)

            title = info.get(
                "title",
                f"video_{counter}"
            )

            title_stub = clean_filename(title)

            file_stub = f"{title_stub}_{lang}"

            log_row["title"] = title
            log_row["title_stub"] = title_stub
            log_row["file_stub"] = file_stub

            print(f"Title: {title}")
            print(f"Lang: {lang}")
            print(f"Output filename stub: {file_stub}")

            # -----------------------------------------
            # DOWNLOAD AUDIO
            # -----------------------------------------

            try:

                download_mp3_low_quality(
                    url,
                    file_stub
                )

                log_row["audio_status"] = "downloaded"

                print("Audio downloaded successfully.")

            except Exception as e:

                log_row["audio_status"] = "error"
                log_row["error_audio"] = str(e)

                print(f"Audio download error:\n{e}")

        except Exception as e:

            log_row["audio_status"] = "error"
            log_row["error_audio"] = f"metadata error: {e}"

            print(f"Metadata extraction error:\n{e}")

        log_rows.append(log_row)

        # -----------------------------------------
        # SAVE LOG AFTER EACH VIDEO
        # -----------------------------------------

        log_df = pd.DataFrame(log_rows)

        log_df.to_excel(
            LOG_FILE,
            index=False
        )

        print(f"Log updated: {LOG_FILE}")

        time.sleep(SLEEP_SECONDS)

    print("\n" + "=" * 70)
    print("DONE")
    print(f"Final log saved to:\n{LOG_FILE}")


# =========================================================
# ENTRYPOINT
# =========================================================

if __name__ == "__main__":
    main()


Checking cookies file:
/content/drive/MyDrive/!records/cookies.txt
cookies.txt found. Size: 494414 bytes

Checking Deno runtime:
/root/.deno/bin/deno
Deno found at: /root/.deno/bin/deno

Loading file: /content/drive/MyDrive/!records/base.xlsx

Unique raw task values:
[nan, 1.0]

Unique raw lang values:
['ru', 'ua', nan]

Total rows in sheet: 4041
Rows selected for processing where task == 1: 324

[1/324] Processing:
https://www.youtube.com/watch?v=BGPlz5ultrc
Title: Таємне життя ТОПів київського метро: фантомні маєтки, фіктивні розлучення та "подарунки" на мільйон
Lang: ua
Output filename stub: Таємне життя ТОПів київського метро_ фантомні маєтки, фіктивні розлучення та _подарунки_ на мільйон_ua
[youtube] Extracting URL: https://www.youtube.com/watch?v=BGPlz5ultrc
[youtube] BGPlz5ultrc: Downloading webpage
[youtube] BGPlz5ultrc: Downloading web client config
[youtube] BGPlz5ultrc: Downloading web player API JSON
[youtube] BGPlz5ultrc: Downloading player 9fc68080-main
[youtube] [jsc:de